# Project Track 2 - Physics-Guided Coordinate DNN

**Choose one variant:**

- **2A Wall-weighted learning:** give near-wall points greater importance.
- **2B Divergence-penalized learning:** add an automatic-differentiation penalty for incompressibility.

The baseline network is provided. Your project is an ablation study: only the physics term changes.

## Required files
`P2_Physics_Guided_DNN.ipynb`, `w4utils.py`, `w5_common.py`, `cavity_data.npz`

<!-- MIE690A enriched learner edition v2 -->

## How to learn from this notebook

This is a guided computational laboratory, not a script to execute without reading. For every numbered stage:

1. read the physical question and write a prediction;
2. inspect the inputs, outputs, units, and split before running code;
3. run the cell and check assertions/warnings;
4. compare with the stated baseline or physical diagnostic; and
5. write one or two sentences explaining what the result does **and does not** establish.

Use **Restart and Run All** before treating any output as final. Hidden state from out-of-order execution is a reproducibility failure.

### Evidence contract

Keep four kinds of evidence separate:

- **numerical evidence:** residuals, accepted cases, data hashes, grid/time/particle budgets;
- **statistical evidence:** losses, relative errors, variability across seeds/cases;
- **physical evidence:** centerlines, walls, divergence, vortex structure, positivity, moments;
- **computational evidence:** runtime, memory, saved configuration, and machine-readable metrics.

A claim is only as strong as the weakest relevant layer.


## Learning objectives and controlled-ablation contract

This notebook asks whether changing the loss improves a named physical diagnostic. Architecture, data, split, optimizer, and training budget remain fixed. You should be able to derive the added term, explain where it is evaluated, select a weight under a global-error constraint, and distinguish a useful tradeoff from a universal improvement claim.

Prerequisites: coordinate DNNs, automatic differentiation, boundary conditions, discrete divergence, and validation-only model selection.


In [ ]:
# Repository/Colab bootstrap. Run this before the import cell below.
from pathlib import Path
import sys

def _find_course_root(start=Path.cwd()):
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "common" / "w5_common.py").exists():
            return base
    # Colab flat-upload fallback: helper files and data beside the notebook.
    if (start / "w5_common.py").exists():
        return start
    raise FileNotFoundError(
        "Course root not found. Clone the repository, or upload w4utils.py, "
        "w5_common.py, the track-specific helpers, and cavity_data.npz as listed above."
    )

COURSE_ROOT = _find_course_root()
COMMON_DIR = COURSE_ROOT / "common" if (COURSE_ROOT / "common").exists() else COURSE_ROOT
DATASET_PATH = COURSE_ROOT / "data" / "cavity_data.npz"
if not DATASET_PATH.exists():
    DATASET_PATH = COURSE_ROOT / "cavity_data.npz"
sys.path.insert(0, str(COMMON_DIR))
print("Course root:", COURSE_ROOT)
print("Common helpers:", COMMON_DIR)
print("Dataset:", DATASET_PATH)


In [ ]:
import time, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
import w4utils, w5_common
importlib.reload(w4utils); importlib.reload(w5_common)
w5_common.set_global_seed(690)
assert w4utils.W4_UTILS_VERSION == "6.3", "Upload the revised w4utils.py (v6.3)."
assert w5_common.W5_COMMON_VERSION == "1.2", "Upload the revised w5_common.py (v1.2)."
data=w5_common.require_week4_files(str(DATASET_PATH))
TRAIN_RE=[100,150,200,225,250,350,400]
VAL_RE=300; TEST_RE=[175,275,375]
VARIANT="2A"  # EDIT to 2B only if approved.


## From data loss to a physics-guided objective

The baseline minimizes standardized pointwise data error. Variant 2A changes sample importance near walls; Variant 2B penalizes the divergence of the network's physical velocity output.

\[
\mathcal L=\mathcal L_{data}+\lambda_{phys}\mathcal L_{phys}.
\]

This is a *soft constraint*: a nonzero weight encourages behavior but does not guarantee it. Large weights can damage global accuracy or make optimization ill-conditioned. The scientific object is therefore a tradeoff curve, not the strongest possible weight.

The two lid corners are excluded from wall RMS because stationary side walls meet a moving lid discontinuously. Penalizing an idealized discontinuity too strongly can force the model to smooth an incompatible target.


## 1. Baseline data and near-wall coordinate

The distance to the nearest wall is

\[d_w=\min(x,1-x,y,1-y).\]

A wall-weighted loss uses a larger sample weight when `d_w` is small. This changes the optimization objective, not the CFD data.

For all wall metrics in this notebook, the four corner points are excluded. The moving lid and stationary side walls impose a discontinuous tangential velocity at the two upper corners, so counting those points would create a nonzero wall-error floor even for the CFD truth.

In [ ]:
train_mask=w5_common.re_mask(data,TRAIN_RE)
val_mask=w5_common.re_mask(data,[VAL_RE])
Xtr,Ytr,_=w5_common.point_samples(data,train_mask,stride=2)
Xva,Yva,_=w5_common.point_samples(data,val_mask,stride=2)
scalers=w5_common.fit_standardizers(Xtr,Ytr)

def wall_distance(X):
    x,y=X[:,1],X[:,2]
    return np.minimum.reduce([x,1-x,y,1-y])

def wall_weights(X,amplitude=0.0,thickness=0.10):
    d=wall_distance(X)
    return 1.0+amplitude*np.exp(-(d/thickness)**2)

plt.hist(wall_distance(Xtr),bins=30); plt.xlabel("distance to nearest wall"); plt.show()


## Variant 2A: how wall weighting changes the question

The distance `d_w` is geometric. A weight such as `1 + A exp(-d_w/delta)` increases the contribution of near-wall samples without adding new physical data. Check the minimum, maximum, and spatial distribution of weights before training.

**Prediction prompt:** As amplitude grows, which response is most plausible: monotonic improvement of every metric, wall improvement with interior degradation, or no effect? State why.


## 2A. Wall-weighted loss

The implementation is intentionally complete. Your responsibility is to choose a **small, predeclared** set of amplitudes, compare them with amplitude zero, and explain the tradeoff between wall and interior accuracy.

In [ ]:
def fit_weighted(amplitude,seed=690):
    model=w5_common.make_dense_model(3,3,(64,64,64),"tanh",seed)
    model.compile(tf.keras.optimizers.Adam(1e-3),loss="mse")
    sw=wall_weights(Xtr,amplitude,thickness=0.10)
    hist=model.fit(scalers.transform_x(Xtr),scalers.transform_y(Ytr),sample_weight=sw,
        validation_data=(scalers.transform_x(Xva),scalers.transform_y(Yva)),
        epochs=900,batch_size=512,verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=65,
                                                     restore_best_weights=True)])
    return {"model":model,"scalers":scalers,"history":hist.history,"amplitude":amplitude}

WALL_AMPLITUDES=[0.0,2.0,8.0]  # Declare before viewing blind tests.


## Variant 2B: derivatives must be in physical units

The network is trained on standardized inputs and outputs, but incompressibility is defined for physical `u`, `v`, `x`, and `y`. Chain-rule scale factors therefore matter. A divergence computed directly in standardized coordinates can be numerically small while representing the wrong physical derivative.

Automatic differentiation measures the continuous derivative of the neural approximation. The CFD labels and reported diagnostic may use a discrete finite-difference operator. Agreement is expected only up to representation and discretization differences.


## 2B. Divergence-penalized loss

The network predicts standardized outputs, but divergence must be computed from physical `u` and `v`. Automatic differentiation evaluates

\[\mathcal L_{div}=\langle(\partial u/\partial x+\partial v/\partial y)^2\rangle.\]

The class below provides the training step. Your project varies only `lambda_div`.

In [ ]:
class DivPenaltyTrainer(tf.keras.Model):
    def __init__(self,net,scalers,lambda_div):
        super().__init__(); self.net=net; self.s=scalers; self.lambda_div=float(lambda_div)
        self.total_tracker=tf.keras.metrics.Mean(name="loss")
        self.data_tracker=tf.keras.metrics.Mean(name="data_loss")
        self.div_tracker=tf.keras.metrics.Mean(name="div_loss")
    @property
    def metrics(self): return [self.total_tracker,self.data_tracker,self.div_tracker]
    def call(self,x,training=False): return self.net(x,training=training)
    def compute_losses(self,Xs,Ys,training):
        X=tf.cast(Xs,tf.float32); Y=tf.cast(Ys,tf.float32)
        with tf.GradientTape(persistent=True) as tape:
            tape.watch(X)
            P=self.net(X,training=training)
            data_loss=tf.reduce_mean(tf.square(P-Y))
            u=P[:,0]*tf.cast(self.s.ystd[0],P.dtype)+tf.cast(self.s.ymean[0],P.dtype)
            v=P[:,1]*tf.cast(self.s.ystd[1],P.dtype)+tf.cast(self.s.ymean[1],P.dtype)
        du=tape.gradient(u,X)[:,1]/tf.cast(self.s.xstd[1],X.dtype)
        dv=tape.gradient(v,X)[:,2]/tf.cast(self.s.xstd[2],X.dtype)
        div_loss=tf.reduce_mean(tf.square(du+dv))
        return data_loss+self.lambda_div*div_loss,data_loss,div_loss
    def train_step(self,data):
        X,Y=data
        with tf.GradientTape() as tape:
            total,datal,divl=self.compute_losses(X,Y,True)
        grads=tape.gradient(total,self.net.trainable_variables)
        self.optimizer.apply_gradients(zip(grads,self.net.trainable_variables))
        for m,v in zip(self.metrics,[total,datal,divl]): m.update_state(v)
        return {m.name:m.result() for m in self.metrics}
    def test_step(self,data):
        total,datal,divl=self.compute_losses(data[0],data[1],False)
        for m,v in zip(self.metrics,[total,datal,divl]): m.update_state(v)
        return {m.name:m.result() for m in self.metrics}

def fit_divergence(lambda_div,seed=690):
    net=w5_common.make_dense_model(3,3,(64,64,64),"tanh",seed)
    trainer=DivPenaltyTrainer(net,scalers,lambda_div)
    trainer.compile(tf.keras.optimizers.Adam(8e-4))
    hist=trainer.fit(scalers.transform_x(Xtr),scalers.transform_y(Ytr),
        validation_data=(scalers.transform_x(Xva),scalers.transform_y(Yva)),
        epochs=700,batch_size=512,verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=55,
                                                     restore_best_weights=True)])
    return {"model":net,"scalers":scalers,"history":hist.history,"lambda_div":lambda_div}

DIV_WEIGHTS=[0.0,1e-4,1e-3]  # Keep the search small and declared in advance.


## 3. Run the selected ablation and choose using validation only

The physics metric is optimized **subject to a global-error tolerance**. This prevents the automatic selector from always choosing the strongest physics weight when it improves divergence or wall error by damaging the field everywhere else. Declare the tolerance before inspecting blind cases.

In [ ]:
settings=WALL_AMPLITUDES if VARIANT=="2A" else DIV_WEIGHTS
GLOBAL_ERROR_TOL=0.10  # EDIT only before blind tests: allow at most 10% validation-error degradation.
models={}; sel=[]
for value in settings:
    t0=time.time()
    b=fit_weighted(value) if VARIANT=="2A" else fit_divergence(value)
    pred=w5_common.predict_case(b,VAL_RE,data["x"],data["y"])
    rep=w5_common.evaluate_prediction(data,VAL_RE,pred)
    sel.append({"setting":float(value),"val_relative_L2_uv":rep["relative_L2_uv"],
                "val_wall_rms":rep["wall_rms_error"],"val_div_l2":rep["div_l2_pred"],
                "seconds":time.time()-t0})
    models[float(value)]=b
selection=pd.DataFrame(sel)

# Reference row: the CFD truth should have zero prediction error and essentially
# zero wall/divergence error under the same metric definitions.
ival=int(np.where(data["Re"].astype(float)==float(VAL_RE))[0][0])
truth=(data["u"][ival],data["v"][ival],data["p"][ival])
truth_rep=w5_common.evaluate_prediction(data,VAL_RE,truth)
reference=pd.DataFrame([{"setting":"CFD truth reference",
    "val_relative_L2_uv":truth_rep["relative_L2_uv"],
    "val_wall_rms":truth_rep["wall_rms_error"],
    "val_div_l2":truth_rep["div_l2_pred"],"seconds":0.0}])
display(pd.concat([reference,selection],ignore_index=True))

baseline_error=float(selection.loc[np.isclose(selection["setting"],0.0),
                                   "val_relative_L2_uv"].iloc[0])
limit=(1.0+GLOBAL_ERROR_TOL)*baseline_error
eligible=selection[selection["val_relative_L2_uv"]<=limit].copy()
metric="val_wall_rms" if VARIANT=="2A" else "val_div_l2"
BEST=float(eligible.sort_values(metric).iloc[0]["setting"])
print(f"Baseline validation L2 = {baseline_error:.5g}")
print(f"Allowed validation L2 limit = {limit:.5g}")
print("Eligible settings:",eligible["setting"].tolist())
print("Frozen setting:",BEST)

fig,ax=plt.subplots(figsize=(5.6,4.1))
ax.scatter(selection["val_relative_L2_uv"],selection[metric],s=55)
for _,row in selection.iterrows():
    ax.annotate(f'{row["setting"]:g}',(row["val_relative_L2_uv"],row[metric]),xytext=(4,4),textcoords="offset points")
ax.axvline(limit,linestyle="--",label="global-error tolerance")
ax.set(xlabel="validation relative L2 velocity error",ylabel=metric,title="Validation tradeoff")
ax.grid(alpha=.3);ax.legend();plt.show()


## How to interpret a Pareto tradeoff

A setting is scientifically useful when its intended metric improves and the cost in other relevant metrics remains within the frozen tolerance. Do not collapse wall, divergence, velocity, and pressure into one unmotivated score.

Plot each candidate in a plane such as global velocity error versus wall RMS or divergence RMS. A dominated point is worse on both axes. A nondominated point may still be unacceptable if it violates the predeclared tolerance.


## 4. Blind comparison and tradeoff analysis

A physics term is useful only if its intended physical metric improves without unacceptable degradation elsewhere.

In [ ]:
rows=[]; stored={}
compare_settings=[0.0] if np.isclose(BEST,0.0) else [0.0,BEST]
for r in TEST_RE:
    stored[r]={}
    for value in compare_settings:
        label="baseline" if np.isclose(value,0.0) else f"{VARIANT}={value:g}"
        pred=w5_common.predict_case(models[float(value)],r,data["x"],data["y"])
        stored[r][label]=pred
        rows.append({"variant":VARIANT,"method":label,"Re":r,
                     **w5_common.evaluate_prediction(data,r,pred)})
results=w5_common.results_frame(rows); results.to_csv("P2_results.csv",index=False)
display(results)
fig=w5_common.plot_case_evidence(data,275,stored[275],f"Track {VARIANT}: physics tradeoff")
plt.show()


## Required report evidence

- State the composite loss and every weight tried.
- Explain how the wall mask or automatic derivative is computed.
- Report global velocity error, wall error, divergence, both centerlines, and pressure error.
- Include the validation tradeoff plot and state the predeclared global-error tolerance.
- Identify a setting that is too weak and one that is too strong.
- State whether the physics term improved **generalization**, merely changed training behavior, or created a new tradeoff.

Do not change architecture, split, learning rate, and physics weight simultaneously.
- Confirm that the CFD-truth reference has essentially zero wall error under the corner-excluded metric.

## Optional stretch extensions (advanced / prize-track only)

Complete the required project first. With instructor approval, choose at most one:

1. Predict a streamfunction and obtain velocity through automatic differentiation, producing an exactly divergence-free parameterization; compare it fairly with the penalty model.
2. Combine wall weighting and divergence regularization in a two-factor ablation while preserving the global-error tolerance.
3. Compare constant and scheduled physics weights and explain whether optimization path, not only final weight, changes the result.


## Concept check and further reading

1. Why is a physics-guided loss not the same as a physics-constrained architecture?
2. Where do normalization scale factors enter the divergence derivative?
3. Why must the zero-weight baseline be retrained under the same protocol?
4. What would make a 5% wall improvement scientifically irrelevant?
5. How would a streamfunction-output architecture change the divergence question?

Read Raissi et al. (2019), Karniadakis et al. (2021), and Cai et al. (2021). For a flow-specific zonal-loss example, see Roohi & Mahdavi (2026) in the annotated references.


## Reproducibility record

Before closing the notebook, record:

- Python and package versions;
- dataset hash and helper versions;
- every physical case in development, validation, and blind sets;
- every seed and candidate value tried;
- the selection rule and when it was frozen;
- output filenames and units; and
- any cell that was skipped, changed, or run with a reduced budget.

Restart the kernel and run all cells in order. If the result changes materially, report the variability instead of selecting the preferred run.

## Troubleshooting without corrupting the experiment

| Symptom | Safe action | Unsafe action |
| --- | --- | --- |
| Missing helper/data file | Re-run the bootstrap and verify paths/hash | Download an unlabeled older copy |
| Training is slow | Use the documented smoke configuration, then label it “smoke” | Quietly reduce epochs/data in the final claim |
| Validation is poor | Inspect scaling, split, and baseline; revise on development data | Open the blind case to choose settings |
| Blind result fails | Report/localize failure and propose a new future experiment | Tune on the blind case while keeping its label “blind” |
| Stochastic result changes | Run multiple declared seeds and report mean/spread | Keep rerunning until one result looks good |
| A neural model loses to interpolation | Verify fairness, then recommend the simpler method | Hide the baseline |

## Final report outline

1. **Question and hypothesis** — one falsifiable sentence.
2. **Data and split** — physical cases, numerical source, and blind unit.
3. **Baseline** — simplest credible comparator using the same allowed information.
4. **Modification** — the one controlled change.
5. **Selection** — validation-only candidates and frozen rule.
6. **Blind numerical result** — aggregate errors and variability.
7. **Physical result** — at least two diagnostics tied to the flow.
8. **Failure or limitation** — where confidence ends.
9. **Cost and reproducibility** — runtime, environment, seeds, saved files.
10. **Conclusion** — helped, hurt, or revealed a tradeoff; no forced positive AI claim.


## Article-output contract

<!-- MIE690A article-aligned validation v3 -->

**Role:** Foundational or supporting notebook; see ARTICLE_FIGURE_MAP.md for its evidence dependency.

All manuscript-facing figures must be generated from retained numerical/model outputs through the documented notebook or shared helper, saved under `results/`, and accompanied by machine-readable metrics. Do not redraw curves by eye or substitute a screenshot for a solver-to-reference comparison. The complete ownership table and exact output filenames are in [`ARTICLE_FIGURE_MAP.md`](../ARTICLE_FIGURE_MAP.md).
